## Архитектура и реализация Document Object Model (DOM)

**DOM** - это независимый от платформы и языка программирования интерфейс (API), интерпретирующий HTML или XML как логическую древовидную структуру. Процесс рендеринга начинается с загрузки HTML в локальную память и автоматического построения дерева, где каждый элемент становится объектом.

**Классификация узлов в DOM-дереве**  
В основе модели лежит иерархия, где корневым узлом для HTML-документа всегда является элемент `<html>`.

<img src="attachments/dom_tree.png" width="1024"/>

|Тип узла|Описание и характеристики|Примеры|
|:--|:--|:--|
|**Document Node**|Корневой объект, представляющий весь документ. Точка входа в API.|`document`|
|**Element Node**|Структурные блоки (HTML-теги). Могут иметь вложенные узлы.|`<div>`, `<section>`, `<a>`|
|**Attribute Node**|Свойства элементов. Не считаются прямыми потомками в дереве.|`id`, `class`, `src`|
|**Text Node**|Содержит текст, включая пробелы и переносы строк.|`"Привет"`, `"\n"`|
|**Comment Node**|Комментарии в коде, доступные программно, но не отображаемые визуально.|`<!-- текст -->`|

**Обработка текстовых фрагментов и пробелов**  
Важной особенностью DOM является превращение пробелов, табуляций и переносов строк в текстовые узлы. Однако существуют исключения:
* До тега `<head>`: Пробелы игнорируются по историческим причинам.
* После тега `</body>`: Любой контент (включая пробелы) автоматически перемещается внутрь body в конец списка дочерних элементов, согласно требованиям спецификации HTML.
___

## Классификация элементов по модели отображения

Визуальное построение страницы базируется на концепции CSS-боксов, определяющих взаимодействие элементов в нормальном потоке (normal flow).  

<img src="attachments/block_vs_inline_elements.png" width='1024'/>

**Блочные элементы (block-level elements)**  
Формируют структурный каркас страницы и участвуют в блочном формате потока (block formatting context).  
* **Поведение в потоке**:
  По умолчанию начинаются с новой строки и, если ширина не задана явно, занимают всю доступную ширину родительского контейнера.
* **Геометрия и модель коробки (box model)**:
  Полностью поддерживают `width`, `height`, `margin`, `padding`, `border`. Эти свойства напрямую влияют на итоговые размеры и позиционирование элемента.
* **Схлопывание внешних отступов (margin collapsing)**:
  Вертикальные `margin` соседних блочных элементов могут схлопываться - используется наибольшее значение, а не сумма.
* **Примеры**:
  `<div>`, `<p>`, `<h1>`–`<h6>`, `<ul>`, `<ol>`, `<section>`, `<article>`, `<header>`, `<footer>`.

**Строчные элементы (inline-level elements)**  
Располагаются внутри строки и участвуют в inline formatting context.  
* **Поведение в потоке**:
  Не начинают новую строку и занимают только ширину, необходимую для отображения содержимого.
* **Ограничения размеров**:
  Свойства `width` и `height` к классическим строчным элементам не применяются. Размер определяется содержимым, шрифтовыми метриками и `line-height`.
* **Особенности отступов**:
  Горизонтальные `margin` и `padding` работают стандартно.
  Вертикальные `margin` не влияют на положение соседних строк.
  Вертикальные `padding` и `border` визуально увеличивают область элемента, но не изменяют высоту строки, что может привести к наложению.
* **Примеры**:
  `<span>`, `<a>`, `<strong>`, `<em>`, `<img>`, `<input>`.

**Элементы с `display: inline-block`**  
`inline-block` - это значение свойства `display`, сочетающее характеристики строчного и блочного форматирования, позволяя элементу оставаться в строке, но при этом вести себя как блочный контейнер в отношении управления размерами и отступами. Это оптимальное решение для навигационных панелей и сеток.  
* **Поведение в потоке**:
  Элемент располагается в строке (как inline) и не начинает новую строку.
* **Геометрия**:
  Поддерживает `width`, `height`, `margin`, `padding`, `border` (как block).
* **Отступы**:
  Вертикальные и горизонтальные отступы работают полноценно - без ограничений, характерных для inline.
* **Особенность выравнивания**:
  По умолчанию выравнивается по базовой линии строки (`baseline`). Это может приводить к появлению "лишнего" вертикального зазора. Управляется через `vertical-align`.
* **Практическое применение**:
  Кнопки, карточки, элементы навигации, бейджи - когда требуется управлять размерами элемента, но сохранить его размещение в строке.

**Сравнительная таблица**
| Характеристика                              | Block                           | Inline                                            | Inline-block |
| ------------------------------------------- | ------------------------------- | ------------------------------------------------- | ------------ |
| Начинается с новой строки                   | Да                              | Нет                                               | Нет          |
| Занимает всю ширину контейнера по умолчанию | Да                              | Нет                                               | Нет          |
| Можно задать `width` и `height`             | Да                              | Нет                                               | Да           |
| Горизонтальные `margin`/`padding`           | Да                              | Да                                                | Да           |
| Вертикальные `margin`                       | Да (возможен margin collapsing) | Не влияют на поток                                | Да           |
| Вертикальные `padding`                      | Да                              | Визуально применяются, но не меняют высоту строки | Да           |
| Участвует в схлопывании margin              | Да                              | Нет                                               | Нет          |
| Располагается в строке с текстом            | Нет                             | Да                                                | Да           |

___

## HTML-атрибуты: Синтаксис и управление состоянием

Атрибуты настраивают поведение элементов и передают дополнительные данные. Они всегда располагаются в открывающем теге в формате `имя="значение"`.  

<img src="attachments/html_attributes.png" width="1024">

**Глобальные и специфические атрибуты**:

* **Глобальные**: Применимы ко всем тегам (`id`, `class`, `style`, `title`). Особняком стоят `data-*` атрибуты, предназначенные для хранения пользовательских данных, доступных скриптам.
* **Специфические**: Предназначены для конкретных задач (например, `href` для ссылок, `src` и `alt` для изображений, `type` для полей ввода).

**Логические (Boolean) атрибуты**:  
Состояние таких атрибутов (истина/ложь) определяется их фактическим присутствием в коде.

* Если атрибут (`disabled`, `checked`, `required`) указан - значение считается `true`.
* **Важно**: Строка `"false"` внутри булевого атрибута (например, `checked="false"`) все равно интерпретируется как истина.
___

## Стратегии поиска и навигации в DOM-дереве

Выбор метода поиска элементов определяет производительность и стабильность приложения.

**Методы на основе селекторов (Selectors API)**:

* `querySelector()`: находит **первый подходящий элемент** по CSS-селектору (обход в глубину).
* `querySelectorAll()`: находит **все подходящие элементы**, но возвращает статический список (Static NodeList) - снимок состояния на момент вызова. Если после этого в DOM что-то изменится - список не обновится.

**Традиционные и live методы**:

* `getElementById()`: самый быстрый способ найти элемент по id, оптимизированный браузерами.
* `getElementsByTagName()`: возвращает **live-коллекцию** (HTMLCollection/Live NodeList), которая автоматически обновляется, если в DOM что-то изменится.

**Сравнение коллекций**
|Характеристика|HTMLCollection / Live NodeList|Static NodeList|
|:--|:--|:--|
|**Динамизм**|Обновляется при изменениях DOM|Неизменен после создания|
|**Содержимое**|Только Element nodes|Любые типы узлов|
|**Методы**|Индексация, item()|Индексация, forEach(), item()|
___

## Ключевые HTML-атрибуты и способы поиска элементов на странице

### 1. Базовые HTML-атрибуты (глобальные)

Эти атрибуты встречаются почти у любых тегов и чаще всего используются для поиска элементов.

#### 1.1 `id`

* **Уникальный** в пределах страницы.
* Самый надёжный якорь для поиска.
* Используется в CSS (`#main`) и JS (`document.getElementById()`).

```html
<div id="product-title">iPhone 15</div>
```

* CSS: `#product-title`
* XPath: `//*[@id="product-title"]`

⚠️ Минус: часто генерируется динамически (`id="__next-123abc"`).

#### 1.2 `class`

* Может повторяться.
* Элемент может иметь **несколько классов**.
* Чаще всего используется для стилизации.

```html
<div class="product-card featured large"></div>
```

* CSS: `.product-card`
* CSS: `.product-card.featured`
* XPath: `//*[contains(@class, "product-card")]`

⚠️ Часто классы автогенерируемые (`css-1a2b3c`). Лучше искать по стабильной части.

#### 1.3 `name`

* Часто используется в формах.
* Особенно важен для `<input>`, `<select>`, `<textarea>`.

```html
<input type="text" name="email">
```

* CSS: `[name="email"]`
* XPath: `//*[@name="email"]`

#### 1.4 `data-*` (кастомные атрибуты)

* Часто содержат реальные данные
* Более стабильны, чем классы
* Используются фронтендом для JS

```html
<div data-product-id="12345" data-price="999"></div>
```

* CSS: `[data-product-id]`
* CSS: `[data-product-id="12345"]`
* XPath: `//*[@data-product-id="12345"]`

🔥 В современном вебе - один из лучших вариантов для поиска.

### 2. Специфические атрибуты популярных тегов

#### 2.1 Ссылки `<a>`

##### `href`

```html
<a href="/product/123">Подробнее</a>
```

* CSS: `a[href]`
* CSS: `a[href*="product"]`
* XPath: `//a[contains(@href, "product")]`

##### `title`

Иногда содержит описание.

#### 2.2 Изображения `<img>`

##### `src`

```html
<img src="/images/item.jpg">
```

* CSS: `img[src]`

##### `alt`

Очень полезен для семантики.

```html
<img alt="iPhone 15 Pro">
```

* CSS: `img[alt*="iPhone"]`

#### 2.3 Формы

##### `type`

```html
<input type="hidden">
<input type="email">
```

* CSS: `input[type="hidden"]`

##### `value`

Может содержать данные.

### 3. Атрибуты, важные для SPA (Single Page Application) и динамических сайтов

Современные сайты (React, Vue, Angular) часто используют специальные атрибуты.

#### 3.1 React / Next.js

* `data-testid`
* `data-reactroot`
* `aria-*`

```html
<button data-testid="submit-button">Send</button>
```

🔥 `data-testid` - один из самых стабильных вариантов.

#### 3.2 ARIA-атрибуты

Используются для accessibility, но очень полезны в скрапинге.

```html
<div role="button" aria-label="Close"></div>
```

* CSS: `[role="button"]`
* CSS: `[aria-label="Close"]`

Часто стабильнее, чем классы.

### 4. Поиск элементов: CSS-селекторы

#### 4.1 Базовые

| Селектор | Значение      |
| -------- | ------------- |
| `div`    | по тегу       |
| `.class` | по классу     |
| `#id`    | по id         |
| `*`      | любой элемент |

#### 4.2 По атрибуту

| Селектор          | Значение          |
| ----------------- | ----------------- |
| `[attr]`          | наличие атрибута  |
| `[attr="value"]`  | точное совпадение |
| `[attr*="value"]` | содержит          |
| `[attr^="value"]` | начинается с      |
| `[attr$="value"]` | заканчивается на  |

Пример:

```css
a[href*="/product/"]
```

#### 4.3 Иерархия

| Селектор     | Значение        |
| ------------ | --------------- |
| `div span`   | потомок         |
| `div > span` | прямой потомок  |
| `h2 + p`     | следующий сосед |
| `h2 ~ p`     | все соседи      |

Пример:

```css
.product-card > .price
```

#### 4.4 Псевдоклассы

| Селектор        | Значение       |
| --------------- | -------------- |
| `:first-child`  | первый ребёнок |
| `:last-child`   | последний      |
| `:nth-child(n)` | по индексу     |
| `:not()`        | исключение     |

Пример:

```css
.product-card:nth-child(1)
```

⚠️ Осторожно: структура DOM может меняться.

### 5. XPath (когда он лучше CSS)

Используется в:

* Selenium
* Scrapy
* lxml

#### 5.1 Базовые примеры

```xpath
//div
//div[@class="product"]
//a[contains(@href, "product")]
```

#### 5.2 По тексту (главное преимущество)

```xpath
//button[text()="Купить"]
//div[contains(text(), "Цена")]
```

CSS не умеет искать по тексту, XPath — умеет.

### 6. Что лучше использовать в веб-скрапинге

Приоритет надёжности:

1. ✅ `data-*`
2. ✅ `id` (если не динамический)
3. ✅ `aria-*`
4. ⚠️ `name`
5. ⚠️ `class`
6. ❌ позиция (`nth-child`)
7. ❌ абсолютный XPath

### 7. Стратегии устойчивого поиска

#### 7.1 Избегать:

* Длинных цепочек
* Абсолютных XPath (`/html/body/div[2]/...`)
* Автогенерируемых классов

#### 7.2 Использовать:

* Частичное совпадение атрибутов
* Поиск по стабильным контейнерам
* Комбинирование условий

Пример устойчивого селектора:

```css
div[data-product-id] .price
```

### 8. Особенности динамических сайтов

Если данные загружаются через JS:

* `requests` не увидит контент
* Нужно:

  * Selenium
  * Playwright
  * Или анализ XHR-запросов (лучший способ)

🔥 Часто проще парсить API, чем DOM.

### 9. DevTools как основной инструмент

Используйте:

* Inspect Element
* Copy → Copy selector
* Copy → Copy XPath
* Network → XHR / Fetch

Главный навык - найти **стабильный атрибут**, а не первый попавшийся.

### 10. Мини-чеклист перед написанием парсера

* Есть ли API?
* Есть ли `data-*`?
* Есть ли стабильный `id`?
* Можно ли искать по `aria-*`?
* Не автогенерируется ли `class`?
* Статический или JS-контент?
